# Restricted Hartree-Fock (HF) Method

In this tutorial we will make the first step -- code the restricted HF method.

## Import Section

The only two things we need to import is Psi4 and NumPy. 

In [1]:
import psi4
import numpy as np

def diag_F(F, A, norb):
    F_p = A.dot(F).dot(A)
    e, C_p = np.linalg.eigh(F_p)
    C = A.dot(C_p)
    C_occ = C[:, :norb]
    D = np.einsum('pi,qi->pq', C_occ, C_occ, optimize=True)
    return (C, D)

def scf_rhf(mol, norb):
    
    MAXITER = 100 
    E_conv = 1.0e-5

    wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option('basis'))
    mints = psi4.core.MintsHelper(wfn.basisset())
    I = np.asarray(mints.ao_eri())
    T = np.asarray(mints.ao_kinetic())
    V = np.asarray(mints.ao_potential())
    H = T + V

    S = np.asarray(mints.ao_overlap())
    A = mints.ao_overlap()
    A.power(-0.5, 1.e-16)
    A = np.asarray(A)

    D = diag_F(H, A, norb)[1]

    SCF_E = 0.0
    E_old = 0.0
    E_nuc = mol.nuclear_repulsion_energy()
    
    for scf_iter in range(1, MAXITER + 1):
        J = np.einsum('pqrs,rs->pq', I, D, optimize=True)
        K = np.einsum('prqs,rs->pq', I, D, optimize=True)
        F = H + 2 * J - K

        SCF_E = np.einsum('pq,pq->', (F + H), D, optimize=True) + E_nuc

        if (abs(SCF_E - E_old) < E_conv):
            break
        E_old = SCF_E

        D = diag_F(F, A, norb)[1] 

        if (scf_iter == MAXITER):
            psi4.core.clean()
            raise Exception("Maximum number of SCF iterations exceeded.")

    return SCF_E

In [2]:
psi4.core.clean_options()
psi4.core.clean()
mol = psi4.geometry("""
0 1
He
symmetry c1
""")
psi4.set_options({'basis': 'def2-QZVP',
                  'reference': 'RHF',
                  'scf_type': 'pk'})

wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option('basis'))

nel = wfn.nalpha() + wfn.nbeta()
norb = nel // 2 

print('\nRHF Energy: %.4f [Eh]' % (scf_rhf(mol, norb)))

print('\nPsi4 RHF Reference Energy: %.4f [Eh]'% (psi4.energy('HF/def2-QZVP', molecule = mol)))

   => Loading Basis Set <=

    Name: DEF2-QZVP
    Role: ORBITAL
    Keyword: BASIS
    atoms 1 entry HE         line    40 file /opt/miniconda3/envs/p4env/share/psi4/basis/def2-qzvp.gbs 

   => Loading Basis Set <=

    Name: DEF2-QZVP
    Role: ORBITAL
    Keyword: BASIS
    atoms 1 entry HE         line    40 file /opt/miniconda3/envs/p4env/share/psi4/basis/def2-qzvp.gbs 


RHF Energy: -2.8616 [Eh]

Scratch directory: /tmp/
   => Libint2 <=

    Primary   basis highest AM E, G, H:  6, 6, 3
    Auxiliary basis highest AM E, G, H:  7, 7, 4
    Onebody   basis highest AM E, G, H:  -, -, -
    Solid Harmonics ordering:            Gaussian

*** tstart() called on dyn172-30-17-145.wireless.uwo.pri
*** at Mon Oct  6 12:13:05 2025

   => Loading Basis Set <=

    Name: DEF2-QZVP
    Role: ORBITAL
    Keyword: BASIS
    atoms 1 entry HE         line    40 file /opt/miniconda3/envs/p4env/share/psi4/basis/def2-qzvp.gbs 


         ---------------------------------------------------------
    